<a href="https://colab.research.google.com/github/koushikthadakamalla/foodzoom-order-analysis/blob/main/foodzoom_orders_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [77]:
import pandas as pd

data = {
    "order_id": [1001, 1002, 1003, 1003, 1005, 1006, 1007, 1008],
    "customer": ["Asha", "ravi", "Meena", "Meena", "Kiran", None, "Divya", "ravi"],
    "city":     ["Hyderabad", "hyderabad", "Mumbai", "Mumbai", "Delhi", "Delhi", "Mumbai", "Hyderabad"],
    "amount":   [450, 300, 720, 720, None, 510, 880, 300],
    "status":   ["Delivered", "Delivered", "Delivered", "Delivered", "Cancelled", "Delivered", "Delivered", "Delivered"]
}

df = pd.DataFrame(data)
df

,order_id,customer,city,amount,status
0,1001,Asha,Hyderabad,450.0,Delivered
1,1002,ravi,hyderabad,300.0,Delivered
2,1003,Meena,Mumbai,720.0,Delivered
3,1003,Meena,Mumbai,720.0,Delivered
4,1005,Kiran,Delhi,NaN,Cancelled
5,1006,None,Delhi,510.0,Delivered
6,1007,Divya,Mumbai,880.0,Delivered
7,1008,ravi,Hyderabad,300.0,Delivered


In [78]:
print(df.shape)
print(df.info())
print(df.isna().sum())
print(df.duplicated().sum())

(8, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  8 non-null      int64  
 1   customer  7 non-null      object 
 2   city      8 non-null      object 
 3   amount    7 non-null      float64
 4   status    8 non-null      object 
dtypes: float64(1), int64(1), object(3)
memory usage: 452.0+ bytes
None
order_id    0
customer    1
city        0
amount      1
status      0
dtype: int64
1


In [79]:
df_clean = df.copy()

df_clean["city"] = df_clean["city"].str.strip().str.title()
df_clean["customer"] = df_clean["customer"].str.strip().str.title()

print(df_clean["city"].unique())
df_clean

['Hyderabad' 'Mumbai' 'Delhi']


,order_id,customer,city,amount,status
0,1001,Asha,Hyderabad,450.0,Delivered
1,1002,Ravi,Hyderabad,300.0,Delivered
2,1003,Meena,Mumbai,720.0,Delivered
3,1003,Meena,Mumbai,720.0,Delivered
4,1005,Kiran,Delhi,NaN,Cancelled
5,1006,None,Delhi,510.0,Delivered
6,1007,Divya,Mumbai,880.0,Delivered
7,1008,Ravi,Hyderabad,300.0,Delivered


In [80]:
df["city"].unique()

array(['Hyderabad', 'hyderabad', 'Mumbai', 'Delhi'], dtype=object)

In [81]:
df_clean["city_upper"]= df_clean["city"].str.upper()
df_clean

,order_id,customer,city,amount,status,city_upper
0,1001,Asha,Hyderabad,450.0,Delivered,HYDERABAD
1,1002,Ravi,Hyderabad,300.0,Delivered,HYDERABAD
2,1003,Meena,Mumbai,720.0,Delivered,MUMBAI
3,1003,Meena,Mumbai,720.0,Delivered,MUMBAI
4,1005,Kiran,Delhi,NaN,Cancelled,DELHI
5,1006,None,Delhi,510.0,Delivered,DELHI
6,1007,Divya,Mumbai,880.0,Delivered,MUMBAI
7,1008,Ravi,Hyderabad,300.0,Delivered,HYDERABAD


In [82]:
print("Rows before:", len(df_clean))

df_clean = df_clean.drop_duplicates(subset="order_id", keep="first")
df_clean = df_clean.reset_index(drop=True)

print("Rows after:", len(df_clean))
print("Is order_id unique now?", df_clean["order_id"].is_unique)
df_clean

Rows before: 8
Rows after: 7
Is order_id unique now? True


,order_id,customer,city,amount,status,city_upper
0,1001,Asha,Hyderabad,450.0,Delivered,HYDERABAD
1,1002,Ravi,Hyderabad,300.0,Delivered,HYDERABAD
2,1003,Meena,Mumbai,720.0,Delivered,MUMBAI
3,1005,Kiran,Delhi,NaN,Cancelled,DELHI
4,1006,None,Delhi,510.0,Delivered,DELHI
5,1007,Divya,Mumbai,880.0,Delivered,MUMBAI
6,1008,Ravi,Hyderabad,300.0,Delivered,HYDERABAD


In [83]:
df_clean[df_clean.duplicated(subset=["customer", "amount", "city"], keep=False)]

,order_id,customer,city,amount,status,city_upper
1,1002,Ravi,Hyderabad,300.0,Delivered,HYDERABAD
6,1008,Ravi,Hyderabad,300.0,Delivered,HYDERABAD


In [84]:
df_clean["customer"] = df_clean["customer"].fillna("Unknown")

mask = (df_clean["status"] == "cancelled") & (df_clean["amount"].isna())
df_clean.loc[mask, "amount"] = 0

print(df_clean.isna().sum())
df_clean

order_id      0
customer      0
city          0
amount        1
status        0
city_upper    0
dtype: int64


,order_id,customer,city,amount,status,city_upper
0,1001,Asha,Hyderabad,450.0,Delivered,HYDERABAD
1,1002,Ravi,Hyderabad,300.0,Delivered,HYDERABAD
2,1003,Meena,Mumbai,720.0,Delivered,MUMBAI
3,1005,Kiran,Delhi,NaN,Cancelled,DELHI
4,1006,Unknown,Delhi,510.0,Delivered,DELHI
5,1007,Divya,Mumbai,880.0,Delivered,MUMBAI
6,1008,Ravi,Hyderabad,300.0,Delivered,HYDERABAD


In [85]:
print(df_clean.columns)

Index(['order_id', 'customer', 'city', 'amount', 'status', 'city_upper'], dtype='object')


In [86]:
df_clean["amount"].sum()

np.float64(3160.0)

In [87]:
revenue_by_city = df_clean.groupby("city")["amount"].mean()
print(revenue_by_city)

print("Total of cities:", revenue_by_city.sum())
print("Total of table :", df_clean["amount"].sum())

city
Delhi        510.0
Hyderabad    350.0
Mumbai       800.0
Name: amount, dtype: float64
Total of cities: 1660.0
Total of table : 3160.0


In [88]:
print(df_clean["status"].value_counts())

total_orders = len(df_clean)
cancelled_orders = (df_clean["status"] == "cancelled").sum()
cancel_rate = cancelled_orders / total_orders * 100

print("Cancelled:", cancelled_orders, "of", total_orders)
print("Cancellation rate:", round(cancel_rate, 1), "%")

df_clean["is_cancelled"] = (df_clean["status"] == "Cancelled").astype(int)
print(df_clean.groupby("city")["is_cancelled"].sum())
print(df_clean.groupby("city")["is_cancelled"].mean() * 100)

status
Delivered    6
Cancelled    1
Name: count, dtype: int64
Cancelled: 0 of 7
Cancellation rate: 0.0 %
city
Delhi        1
Hyderabad    0
Mumbai       0
Name: is_cancelled, dtype: int64
city
Delhi        50.0
Hyderabad     0.0
Mumbai        0.0
Name: is_cancelled, dtype: float64


In [89]:
delivered = df_clean[df_clean["status"] == "Delivered"]
average_delivered_orders= delivered.groupby("city")["amount"].mean()
print(average_delivered_orders)

city
Delhi        510.0
Hyderabad    350.0
Mumbai       800.0
Name: amount, dtype: float64


In [90]:
import sqlite3

conn = sqlite3.connect(":memory:")
df_clean.to_sql("orders", conn, index=False, if_exists="replace")

query = """
SELECT city,
       COUNT(*)          AS total_orders,
       SUM(is_cancelled) AS cancelled_orders,
       SUM(amount)       AS revenue
FROM orders
GROUP BY city
ORDER BY revenue DESC
"""
result =pd.read_sql(query,conn)

In [91]:
result.style.hide(axis="index")

city,total_orders,cancelled_orders,revenue
Mumbai,2,0,1600.000000
Hyderabad,3,0,1050.000000
Delhi,2,1,510.000000


In [92]:
pd.read_sql("SELECT * FROM orders WHERE customer = 'Ravi'", conn)
pd.read_sql("SELECT * FROM orders WHERE customer = 'Unknown'", conn)

,order_id,customer,city,amount,status,city_upper,is_cancelled
0,1006,Unknown,Delhi,510.0,Delivered,DELHI,0


In [93]:
pd.read_sql("SELECT* , AVG(amount) FROM orders WHERE status = 'Delivered'" , conn)

,order_id,customer,city,amount,status,city_upper,is_cancelled,AVG(amount)
0,1001,Asha,Hyderabad,450.0,Delivered,HYDERABAD,0,526.666667


In [94]:
print(df_clean["customer"].unique())
pd.read_sql("SELECT DISTINCT customer FROM orders", conn)

['Asha' 'Ravi' 'Meena' 'Kiran' 'Unknown' 'Divya']


,customer
0,Asha
1,Ravi
2,Meena
3,Kiran
4,Unknown
5,Divya


In [95]:
query2 = """
SELECT city,
       AVG(amount) AS avg_order_value
FROM orders
WHERE status = 'Delivered'
GROUP BY city
"""
pd.read_sql(query2, conn)

,city,avg_order_value
0,Delhi,510.0
1,Hyderabad,350.0
2,Mumbai,800.0


In [96]:
  print(pd.read_sql("SELECT * FROM orders WHERE customer = 'Ravi'", conn))
  print(pd.read_sql("SELECT * FROM orders WHERE customer = 'Unknown'", conn))

   order_id customer       city  amount     status city_upper  is_cancelled
0      1002     Ravi  Hyderabad   300.0  Delivered  HYDERABAD             0
1      1008     Ravi  Hyderabad   300.0  Delivered  HYDERABAD             0
   order_id customer   city  amount     status city_upper  is_cancelled
0      1006  Unknown  Delhi   510.0  Delivered      DELHI             0


In [97]:
import random

random.seed(42)

topic = []

def producer(n):
    cities = ["Hyderabad", "Mumbai", "Delhi"]
    statuses = ["Delivered", "Delivered", "Delivered", "Cancelled"]
    for i in range(n):
        order = {
            "order_id": 2001 + i,
            "city": random.choice(cities),
            "amount": random.randint(200, 900),
            "status": random.choice(statuses),
        }
        topic.append(order)

producer(10)
print("Messages on the belt:", len(topic))

revenue = {}
cancelled = {}
offset = 0

while offset < len(topic):
    order = topic[offset]
    city = order["city"]

    if order["status"] == "Delivered":
        revenue[city] = revenue.get(city, 0) + order["amount"]
    else:
        cancelled[city] = cancelled.get(city, 0) + 1

    offset = offset + 1

print("Revenue so far:", revenue)
print("Cancellations :", cancelled)
print("Last offset read:", offset)

expected = sum(o["amount"] for o in topic if o["status"] == "Delivered")
notebook_total = sum(revenue.values())
print("Sanity check:", notebook_total == expected)

Messages on the belt: 10
Revenue so far: {'Delhi': 2185, 'Hyderabad': 2013}
Cancellations : {'Delhi': 1, 'Hyderabad': 1}
Last offset read: 10
Sanity check: True


In [98]:
producer(5)
print("Messages in group:", len(topic))
print("My bookmark is at:", offset)

Messages in group: 15
My bookmark is at: 10


In [99]:
while offset < len(topic):
    order = topic[offset]
    city = order["city"]

    if order["status"] == "Delivered":
        revenue[city] = revenue.get(city, 0) + order["amount"]
    else:
        cancelled[city] = cancelled.get(city, 0) + 1

    offset = offset + 1

print("Revenue so far:", revenue)
print("Cancellations :", cancelled)
print("Last offset read:", offset)

Revenue so far: {'Delhi': 2185, 'Hyderabad': 3816, 'Mumbai': 1036}
Cancellations : {'Delhi': 1, 'Hyderabad': 1}
Last offset read: 15


In [100]:
producer(5)
print("Messages in group:", len(topic))
print("My bookmark is at:", offset)

Messages in group: 20
My bookmark is at: 15


In [101]:
while offset < len(topic):
    order = topic[offset]
    city = order["city"]

    if order["status"] == "Delivered":
        revenue[city] = revenue.get(city, 0) + order["amount"]
    else:
        cancelled[city] = cancelled.get(city, 0) + 1

    offset = offset +1

print("Revenue so far:", revenue)
print("Cancellations :", cancelled)
print("Last offset read:", offset)

Revenue so far: {'Delhi': 3414, 'Hyderabad': 4693, 'Mumbai': 1597}
Cancellations : {'Delhi': 1, 'Hyderabad': 1}
Last offset read: 20
